# Lab 17 — Gradient Boosting, XGBoost, and Hyperparameter Tuning

In this lab you will fit gradient boosting models directly, compare sklearn's `GradientBoostingClassifier` against `XGBClassifier` head to head, and then tune hyperparameters the honest way — with cross-validation on the training set, saving the test set for a single final check.

**Concepts covered:** sequential residual fitting, the learning-rate/n_estimators tradeoff, XGBoost's second-order (gradient + Hessian) updates and L2 leaf-weight regularization, grid search vs. random search, and why hyperparameter tuning must never touch the test set.

**Reference working sessions:**
- `working-sessions/supervised/09_gradient_boosting.ipynb`
- `working-sessions/supervised/10_xgboost.ipynb`
- `working-sessions/model_evaluation/05_hyperparameter_tuning.ipynb`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import roc_auc_score
import xgboost as xgb

from tkh_utils import (
    PALETTE, FONT, base_layout,
    check_answer, make_answer_key, make_grading_summary,
    load_heart_disease,
)

_ak = make_answer_key({
    'q1': 'B',
    'q2': 'A',
    'q3': 'B',
    'q4': 'C',
})

---
## Section A — Multiple Choice

Fill in each answer variable with the letter of the best answer (A, B, C, or D).

In [ ]:
# Q1 — At every round after the first, what does a new tree in a gradient
# boosting ensemble actually get trained to predict?
#
#   A) The original target values directly, the same target every other
#      tree in the ensemble was trained on
#   B) The residuals (pseudo-residuals) left over from the current
#      ensemble's predictions — how wrong the ensemble still is for each
#      training row
#   C) A random subset of the previous tree's own predictions
#   D) The feature importances computed from the previous tree

q1_answer = "___"  # Replace with A, B, C, or D

assert q1_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q1_answer, _ak['q1']), \
    "Not quite — revisit working-sessions/supervised/09_gradient_boosting.ipynb " \
    "and the \"How it learns\" section."
print("✓ Question 1 correct!")

In [ ]:
# Q2 — Plain gradient boosting only uses the gradient (first-order
# information) to decide each tree's correction. XGBoost also uses the
# Hessian (second-order information). What does the Hessian actually buy it?
#
#   A) It tells the model not just which direction to adjust a leaf's value
#      but how large a step to take before the loss would start curving
#      back up, producing a more precise, better-sized update
#   B) It is only used to speed up training on GPUs and has no effect on
#      accuracy
#   C) It lets XGBoost skip cross-validation entirely
#   D) The gradient and the Hessian are the same quantity in XGBoost — it's
#      just a naming difference

q2_answer = "___"  # Replace with A, B, C, or D

assert q2_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q2_answer, _ak['q2']), \
    "Not quite — revisit working-sessions/supervised/10_xgboost.ipynb " \
    "and the \"How it learns\" / \"What's happening?\" sections."
print("✓ Question 2 correct!")

In [ ]:
# Q3 — XGBoost's optimal leaf weight is w* = -G / (H + lambda), where G and H
# are the summed gradient and Hessian of the training points in that leaf.
# What happens to a leaf's weight as you increase lambda?
#
#   A) lambda has no effect unless gamma is also increased
#   B) The optimal leaf weight shrinks toward zero, making every tree's
#      updates more conservative
#   C) The optimal leaf weight grows without bound, making every tree's
#      updates more aggressive
#   D) lambda only affects which feature is chosen for a split, not the
#      leaf's value

q3_answer = "___"  # Replace with A, B, C, or D

assert q3_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q3_answer, _ak['q3']), \
    "Not quite — revisit working-sessions/supervised/10_xgboost.ipynb " \
    "and the Math Explorer widget's lambda slider."
print("✓ Question 3 correct!")

In [ ]:
# Q4 — Why is it a mistake to pick the best hyperparameter combination by
# checking which one scores highest on the test set?
#
#   A) It isn't a mistake — the test set is exactly what hyperparameters
#      should be tuned against
#   B) It makes the search slower, but the final reported test score is
#      still trustworthy afterward
#   C) It leaks test set information into the model-selection process, so
#      the test score you report afterward is optimistic and no longer an
#      honest estimate of performance on new data
#   D) It only matters for grid search, not for random search or Bayesian
#      optimization

q4_answer = "___"  # Replace with A, B, C, or D

assert q4_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q4_answer, _ak['q4']), \
    "Not quite — revisit working-sessions/model_evaluation/05_hyperparameter_tuning.ipynb " \
    "and the \"How we got here\" section."
print("✓ Question 4 correct!")

In [ ]:
make_grading_summary([
    (q1_answer, _ak['q1'], "Q1: What each new tree in gradient boosting fits"),
    (q2_answer, _ak['q2'], "Q2: What the Hessian buys XGBoost over plain gradient boosting"),
    (q3_answer, _ak['q3'], "Q3: What increasing lambda does to a leaf's optimal weight"),
    (q4_answer, _ak['q4'], "Q4: Why tuning must never be scored against the test set"),
], total=4)

---
## Section B — Coding Exercises

The three exercises below explore the learning-rate/n_estimators tradeoff in plain gradient boosting, compare XGBoost against sklearn's GBM at matched hyperparameters, and then tune a model honestly with cross-validation, touching the test set only once at the very end. Run the setup cell first.

In [ ]:
# Shared setup — run this before the exercises
X, y = load_heart_disease()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Dataset shape:", X.shape)
print("Training samples:", X_train.shape[0], " Test samples:", X_test.shape[0])

### B1 — Learning rate vs. number of estimators

Fit two gradient boosting models on the same data: one with a high learning rate and very few rounds, one with a low learning rate and many more rounds. Compare their test-set ROC-AUC.

In [ ]:
# B1 — Fit a fast (high learning_rate, few rounds) and a slow (low learning_rate, many rounds) model
fast_model = ___(n_estimators=20, learning_rate=0.5, max_depth=3, random_state=42)   # YOUR CODE — the gradient boosting classifier for this exercise
fast_model.fit(___, ___)   # YOUR CODE — the training features and training target

slow_model = ___(n_estimators=300, learning_rate=0.02, max_depth=3, random_state=42)   # YOUR CODE — the gradient boosting classifier for this exercise
slow_model.fit(___, ___)   # YOUR CODE — the training features and training target

fast_test_auc = roc_auc_score(___, fast_model.predict_proba(X_test)[:, 1])   # YOUR CODE — the true test labels
slow_test_auc = roc_auc_score(___, slow_model.predict_proba(X_test)[:, 1])   # YOUR CODE — the true test labels

print(f"Fast model  (n_estimators=20,  learning_rate=0.5):  test AUC = {fast_test_auc:.3f}")
print(f"Slow model  (n_estimators=300, learning_rate=0.02): test AUC = {slow_test_auc:.3f}")

# --- checks ---
assert abs(fast_test_auc - slow_test_auc) < 0.1, \
    "A high learning rate with few rounds and a low learning rate with many rounds should land in a similar ballpark"
assert slow_model.n_estimators > fast_model.n_estimators
print("✓ B1 complete!")

### B2 — sklearn GBM vs. XGBoost, head to head

Fit sklearn's `GradientBoostingClassifier` and XGBoost's `XGBClassifier` with matched `n_estimators`, `learning_rate`, and `max_depth`, and compare their test-set ROC-AUC.

In [ ]:
# B2 — Fit sklearn GBM and XGBoost at matched hyperparameters and compare AUC
gbm_model = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=3, random_state=42)
gbm_model.fit(X_train, y_train)

xgb_model = ___(   # YOUR CODE — the XGBoost classifier for this exercise
    n_estimators=150, learning_rate=0.1, max_depth=3, reg_lambda=1.0,
    eval_metric='logloss', random_state=42, verbosity=0,
)
xgb_model.fit(___, ___)   # YOUR CODE — the training features and training target

gbm_auc = roc_auc_score(y_test, gbm_model.predict_proba(X_test)[:, 1])
xgb_auc = roc_auc_score(___, xgb_model.predict_proba(___)[:, 1])   # YOUR CODE — the true test labels; the test features

print(f"sklearn GBM  test AUC: {gbm_auc:.3f}")
print(f"XGBoost      test AUC: {xgb_auc:.3f}")

# --- checks ---
assert 0.7 < xgb_auc < 1.0, \
    "XGBoost's test AUC should be a plausible score well above chance (0.5) and below a perfect 1.0"
assert xgb_model.get_params()['reg_lambda'] == 1.0
print("✓ B2 complete!")

### B3 — Tuning honestly with GridSearchCV

Search a small grid of `n_estimators` and `max_depth` values using cross-validation on the training set only, then check the best model's score on the test set exactly once, after the search is already finished.

In [ ]:
# B3 — Grid search with cross-validation, test set touched only once at the end
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3, 4],
}
grid_search = ___(   # YOUR CODE — the scikit-learn search class that exhaustively tries every combination in param_grid
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=___,   # YOUR CODE — number of cross-validation folds to use (5)
    scoring='roc_auc',
    n_jobs=-1,
)
grid_search.fit(___, ___)   # YOUR CODE — the training features and training target only, never the test set

print("Best params found via cross-validation:", grid_search.best_params_)
print("Best cross-validated AUC:              ", round(grid_search.best_score_, 3))

# The test set is touched exactly once here, after the search is already finished
final_test_auc = roc_auc_score(___, grid_search.best_estimator_.predict_proba(___)[:, 1])   # YOUR CODE — the true test labels; the test features
print("Final held-out test AUC:               ", round(final_test_auc, 3))

# --- checks ---
assert grid_search.best_score_ > 0.85, \
    "The best cross-validated AUC found by the grid search should be well above chance"
assert final_test_auc > 0.8, \
    "The final held-out test AUC should also be strong, confirming the CV estimate generalized"
print("✓ B3 complete!")

---
## Section C — Applied Problem

A hospital wants to know whether it is worth the extra complexity of tuning XGBoost, or whether a default sklearn GBM would serve just as well. Use `RandomizedSearchCV` to tune XGBoost's `n_estimators`, `max_depth`, `learning_rate`, and `reg_lambda` against cross-validated ROC-AUC on the training set, then compare the tuned model to both untuned defaults on the held-out test set — checked only once, at the end. Fill in the blanks to run the full pipeline.

In [ ]:
# Section C — Tuning XGBoost without touching the test set until the very end

# --- Step 1: Load and split ---
X_c, y_c = load_heart_disease()
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    ___, ___, test_size=___, random_state=___, stratify=___
    # YOUR CODE — split the features and target, holding out 20% of the data
    # with a fixed seed for reproducibility, preserving the class balance
)

# --- Step 2: Random search over XGBoost hyperparameters, scored by cross-validation ---
param_dist = {
    'n_estimators': [50, 100, 150, 200, 300],
    'max_depth': [2, 3, 4, 5],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'reg_lambda': [0.1, 1.0, 3.0, 10.0],
}
random_search = ___(   # YOUR CODE — the scikit-learn search class that samples a fixed number of random combinations instead of trying every one
    xgb.XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
)
random_search.fit(___, ___)   # YOUR CODE — the training features and training target

# --- Step 3: Score the tuned model, and two untuned defaults, on the test set ---
tuned_model = random_search.___   # YOUR CODE — the attribute holding the already-refit best model the search found
tuned_test_auc = roc_auc_score(y_c_test, tuned_model.predict_proba(X_c_test)[:, 1])

default_xgb = xgb.XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0).fit(X_c_train, y_c_train)
default_xgb_auc = roc_auc_score(y_c_test, default_xgb.predict_proba(X_c_test)[:, 1])

default_gbm = GradientBoostingClassifier(random_state=42).fit(X_c_train, y_c_train)
default_gbm_auc = roc_auc_score(y_c_test, default_gbm.predict_proba(X_c_test)[:, 1])

# --- Step 4: Compare all three ---
results_c = pd.DataFrame({
    "model": ["Default sklearn GBM", "Default XGBoost", "Tuned XGBoost (RandomizedSearchCV)"],
    "test_auc": [default_gbm_auc, default_xgb_auc, tuned_test_auc],
})
print(results_c.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=results_c, x="model", y="test_auc", color=PALETTE["primary"], ax=ax)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("Test ROC-AUC")
ax.set_title("Default GBM vs. Default XGBoost vs. Tuned XGBoost")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

# --- checks ---
assert tuned_test_auc >= default_xgb_auc, \
    "The tuned XGBoost model should score at least as well as the default XGBoost model on the held-out test set"
assert tuned_test_auc >= default_gbm_auc - 0.03, \
    "The tuned XGBoost model should be roughly as good as, or better than, the default sklearn GBM"
print("✓ Section C complete!")
print(f"  Default GBM: {default_gbm_auc:.3f}. Default XGBoost: {default_xgb_auc:.3f}. Tuned XGBoost: {tuned_test_auc:.3f}.")

---

## Section D — Reflection

These questions are for reflection. Edit the markdown cells below each question to write your response. There are no wrong answers, we are looking for thoughtful engagement with what you have learned. Your instructor may review these.

**Question D1**

You need a model for a fraud-detection pipeline that retrains every night on fresh data under a tight compute budget. Would you reach for plain gradient boosting or XGBoost? Justify your answer using at least one specific difference between them (regularization, second-order gradients, or training speed) from `working-sessions/supervised/10_xgboost.ipynb`.

*Your response here...*

**Question D2**

You have a budget of only 15 model fits to tune 4 XGBoost hyperparameters (`n_estimators`, `max_depth`, `learning_rate`, `reg_lambda`). Would you reach for grid search, random search, or Bayesian optimization? Justify your choice using the Reference table in `working-sessions/model_evaluation/05_hyperparameter_tuning.ipynb`.

*Your response here...*

**Question D3**

Describe a moment in this course (or elsewhere) where you were tempted to peek at test-set performance while still tuning a model, or where you've seen someone else do it. Knowing what you now know about why tuning must stay on the training/validation side of the split, what would you say to that person — or to yourself in the moment?

*Your response here...*